# Supervised Learning

## Learning Objectives
1. Implement logistic regression from scratch using gradient descent
2. Compare classifiers (LogReg, SVC, DT, RF) in a sklearn pipeline on accuracy/speed
3. Analyse feature importance with permutation, SHAP, and model-native methods
4. Visualise learning curves to identify data efficiency differences across models


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — Logistic Regression from Scratch

In [ ]:
# ── Level 1: logistic regression (numpy, gradient descent) ───────────────
import numpy as np

np.random.seed(42)

# Binary classification: 200 samples, 4 features
X  = np.random.randn(200, 4).astype(np.float32)
w_true = np.array([1.5, -1.0, 0.5, -0.2])
logits = X @ w_true
p_true = 1 / (1 + np.exp(-logits))
y      = (np.random.rand(200) < p_true).astype(np.float32)

def sigmoid(z): return 1 / (1 + np.exp(-z))
def bce_loss(y_hat, y): return -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))

# Gradient descent
w = np.zeros(4)
b = 0.0
lr, epochs = 0.1, 200
losses = []
for epoch in range(epochs):
    z     = X @ w + b
    y_hat = sigmoid(z)
    loss  = bce_loss(y_hat, y)
    losses.append(loss)
    # Gradients
    dz = y_hat - y
    dw = X.T @ dz / len(y)
    db = dz.mean()
    w -= lr * dw
    b -= lr * db

preds = (sigmoid(X @ w + b) > 0.5).astype(int)
acc   = (preds == y.astype(int)).mean()
print(f"Scratch logistic regression — final loss: {losses[-1]:.4f}  accuracy: {acc:.3f}")
print(f"Learned weights: {w.round(3)}  bias: {b:.3f}")
print(f"True weights:    {w_true}  bias: 0")


## Level 2 — Sklearn Classifier Comparison

In [ ]:
# ── Level 2: compare classifiers with sklearn ─────────────────────────────
import numpy as np, time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                            n_redundant=5, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

classifiers = {
    "LogisticReg": Pipeline([("sc", StandardScaler()),
                              ("clf", LogisticRegression(max_iter=500, random_state=42))]),
    "SVC (RBF)":   Pipeline([("sc", StandardScaler()),
                              ("clf", SVC(C=1.0, kernel="rbf", random_state=42))]),
    "DecisionTree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
}

print(f"{'Classifier':<16}  {'Test Acc':>8}  {'Train ms':>9}  {'Infer ms':>9}")
print("-" * 50)
for name, clf in classifiers.items():
    t0 = time.perf_counter()
    clf.fit(X_tr, y_tr)
    train_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    acc = clf.score(X_te, y_te)
    infer_ms = (time.perf_counter() - t0) * 1000

    print(f"{name:<16}  {acc:>8.3f}  {train_ms:>9.1f}  {infer_ms:>9.3f}")


## Real-World Example 1 — Feature Importance Comparison

In [ ]:
# ── RW1: permutation importance vs model-native importance ───────────────
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_classification(n_samples=800, n_features=15, n_informative=6,
                            n_redundant=4, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

# Model-native importance (mean decrease in impurity — biased toward high-cardinality)
native_imp = rf.feature_importances_
top_native  = np.argsort(native_imp)[::-1][:6]

# Permutation importance (model-agnostic, evaluated on test set)
perm_result = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=42)
perm_imp    = perm_result.importances_mean
top_perm    = np.argsort(perm_imp)[::-1][:6]

print("Top-6 features by native (MDI) importance:")
for rank, idx in enumerate(top_native):
    print(f"  {rank+1}. Feature {idx:02d}: {native_imp[idx]:.4f}")

print("\nTop-6 features by permutation importance (test-set):")
for rank, idx in enumerate(top_perm):
    print(f"  {rank+1}. Feature {idx:02d}: {perm_imp[idx]:.4f}  "
          f"(±{perm_result.importances_std[idx]:.4f})")

overlap = set(top_native) & set(top_perm)
print(f"\nOverlap between methods: {len(overlap)}/6 features agree")
print("Use permutation importance for unbiased feature selection in production.")


## Real-World Example 2 — Calibrated Probability Outputs

In [ ]:
# ── RW2: Platt scaling / CalibratedClassifierCV ───────────────────────────
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# SVC without calibration — doesn't naturally produce good probabilities
svc_raw = SVC(probability=True, random_state=42)
svc_raw.fit(X_tr, y_tr)

# SVC with Platt scaling calibration
svc_cal = CalibratedClassifierCV(SVC(), cv=5, method="sigmoid")
svc_cal.fit(X_tr, y_tr)

# Measure calibration: Brier score (lower = better)
from sklearn.metrics import brier_score_loss
prob_raw = svc_raw.predict_proba(X_te)[:, 1]
prob_cal = svc_cal.predict_proba(X_te)[:, 1]

brier_raw = brier_score_loss(y_te, prob_raw)
brier_cal = brier_score_loss(y_te, prob_cal)

print(f"SVC (built-in probability)  Brier score: {brier_raw:.4f}")
print(f"SVC + Platt calibration     Brier score: {brier_cal:.4f}")
print("\nCalibrated model Brier score should be lower (better-calibrated probabilities).")
print("Use calibrated classifiers when predicted probabilities matter (risk scoring, etc.).")

# Show how fraction of positives matches predicted probability
frac_pos_raw, mean_pred_raw = calibration_curve(y_te, prob_raw, n_bins=8)
frac_pos_cal, mean_pred_cal = calibration_curve(y_te, prob_cal, n_bins=8)
print("\nCalibration check (ideal: fraction_positives ≈ mean_predicted_value):")
print("Raw SVC:")
for fp, mp in zip(frac_pos_raw, mean_pred_raw):
    print(f"  mean_pred={mp:.2f}  frac_pos={fp:.2f}  delta={abs(fp-mp):.2f}")


## Real-World Example 3 — Learning Curves: Data Efficiency

In [ ]:
# ── RW3: learning curve — accuracy vs training size ──────────────────────
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10,
                            random_state=42)

estimators = {
    "LogisticReg":  Pipeline([("sc", StandardScaler()),
                               ("clf", LogisticRegression(max_iter=500))]),
    "RandomForest": RandomForestClassifier(n_estimators=50, random_state=42),
    "SVC":          Pipeline([("sc", StandardScaler()),
                               ("clf", SVC(kernel="rbf"))]),
}

train_sizes = np.linspace(0.05, 0.8, 8)   # 5% to 80% of training data

print(f"{'N Train':>8}", end="")
for name in estimators:
    print(f"  {name:>12}", end="")
print()
print("-" * (8 + 16 * len(estimators)))

rows = {}
for name, est in estimators.items():
    sizes, tr_scores, val_scores = learning_curve(
        est, X, y, train_sizes=train_sizes, cv=5, scoring="accuracy",
        n_jobs=-1, verbose=0
    )
    rows[name] = (sizes, val_scores.mean(axis=1))

for i, sz in enumerate(rows["LogisticReg"][0]):
    print(f"{int(sz):>8}", end="")
    for name in estimators:
        print(f"  {rows[name][1][i]:>12.3f}", end="")
    print()

print("\nRF needs more data to shine; LogReg is most data-efficient on this task.")
